### Installation of the Ultralytics Framework

The following command installs the **Ultralytics** package, which provides the implementation framework for YOLO-based computer vision models:

In [ ]:
!pip install ultralytics

### Verification of the Experimental Environment

The following commands are used to verify the installation and inspect the execution environment:

In [ ]:
!pip show ultralytics
!yolo checks
!ls -lah /kaggle/working

### Dataset Configuration

A dataset configuration file is created to define the directory structure and class information required for training and evaluating the object detection model. The configuration specifies the root directory of the **Yellow Onion** dataset and establishes separate paths for the training, validation, and test image sets.

The dataset contains a single object category, **yellow onion**, represented by one class label. The class index is therefore defined as `0`, with the corresponding class name specified as `yellowOnion`. This configuration enables the Ultralytics training pipeline to automatically locate the respective image datasets and associate the annotated objects with the correct class during model training, validation, and testing.


In [ ]:
%%writefile /kaggle/working/data.yaml
path: /kaggle/input/datasets/muhammadibnerafiq/smart-crops/yellowOnion
train: train/images
val: valid/images
test: test/images
nc: 1
names: ['yellowOnion']

### YOLO-Based Instance Segmentation Training

The segmentation model is trained using the Ultralytics YOLO framework with the **YOLO12s segmentation architecture**. The previously defined dataset configuration is provided to specify the training, validation, and test data used during the experiment.

The model is trained for a maximum of **500 epochs** with an input image resolution of **640 × 640 pixels** and a batch size of **16**. Training is performed on the first available GPU (`device=0`) to accelerate model optimization. An early-stopping patience of **50 epochs** is specified, allowing training to terminate when the validation performance does not improve for a sustained number of epochs, thereby reducing unnecessary computation and potential overfitting.

Two worker processes are allocated for data loading, while a fixed random seed of **42** is used to improve reproducibility across experimental runs. The resulting training artifacts, including model checkpoints and evaluation outputs, are stored under the designated segmentation experiment directory. The experiment is assigned a descriptive identifier corresponding to the Yellow Onion segmentation model.

Overall, this configuration establishes a controlled training procedure for learning pixel-level object masks of yellow onions from the annotated dataset.


In [ ]:
!yolo segment train model=yolo12s-seg.yaml data=/kaggle/working/data.yaml epochs=500 imgsz=640 batch=16 device=0 patience=50 workers=2 seed=42 project=/kaggle/working/runs_seg name=yellowOnion12s

### For YOLO26s-seg

In [ ]:
# !yolo segment train model=yolo26s-seg.pt data=/kaggle/working/data.yaml epochs=500 imgsz=640 batch=16 device=0 patience=50 workers=2 seed=42 project=/kaggle/working/runs_seg name=cabbage_yolo26s

### Model Evaluation on the Test Dataset

The trained segmentation model is evaluated using the checkpoint corresponding to the best validation performance. The evaluation is performed on the independent **test split** of the Yellow Onion dataset using the same dataset configuration and an input resolution of **640 × 640 pixels**.

The evaluation is executed on the GPU to accelerate inference. By explicitly selecting the test split, the assessment is conducted on data that is separate from the samples used during model optimization and validation. This provides an unbiased evaluation of the model's ability to generalize to previously unseen images.

The evaluation procedure measures the segmentation model's detection and mask-prediction performance using the metrics provided by the Ultralytics framework. These results are subsequently used to quantify the effectiveness of the trained YOLO-based segmentation model on the Yellow Onion dataset.


In [ ]:
!yolo segment val model=/kaggle/working/runs_seg/yellowOnion12s/weights/best.pt data=/kaggle/working/data.yaml split=test imgsz=640 device=0

### Same for other three crops , Uncomment them and run them to train

In [ ]:
# %%writefile /kaggle/working/data_kale.yaml
# path: /kaggle/input/datasets/muhammadibnerafiq/smart-crops/purpleKale
# train: train/images
# val: valid/images
# test: test/images
# nc: 1
# names: ['purple_kale']

In [ ]:
#!yolo segment train model=yolo12s-seg.yaml data=/kaggle/working/data_kale.yaml epochs=500 imgsz=640 batch=16 device=0 patience=50 workers=2 seed=42 project=/kaggle/working/runs_seg name=kale_yolo12s

In [ ]:
# !yolo segment train model=yolo26s-seg.pt data=/kaggle/working/data_kale.yaml epochs=500 imgsz=640 batch=16 device=0 patience=50 workers=2 seed=42 project=/kaggle/working/runs_seg name=kale_yolo26s

In [ ]:
#!yolo segment val model=/kaggle/working/runs_seg/kale_yolo12s/weights/best.pt data=/kaggle/working/data_kale.yaml split=test imgsz=640 device=0

In [ ]:
# %%writefile /kaggle/working/data_onion.yaml
# path: /kaggle/input/datasets/muhammadibnerafiq/smart-crops/yellowOnion
# train: train/images
# val: valid/images
# test: test/images
# nc: 1
# names: ['yellow_onion']

In [ ]:
# !yolo segment train model=yolo26s-seg.pt data=/kaggle/working/data_onion.yaml epochs=500 imgsz=640 batch=16 device=0 patience=50 workers=2 seed=42 project=/kaggle/working/runs_seg name=onion_yolo26s

In [ ]:
#!yolo segment train model=yolo26s-seg.pt data=/kaggle/working/data_onion.yaml epochs=500 imgsz=640 batch=16 device=0 patience=50 workers=2 seed=42 project=/kaggle/working/runs_seg name=onion_yolo26s

In [ ]:
# !yolo segment val model=/kaggle/working/runs_seg/onion_yolo26s/weights/best.pt data=/kaggle/working/data_onion.yaml split=test imgsz=640 device=0

## Clean Up

In [ ]:
import glob, os

def keep_only_best(root="/kaggle/working"):
    """Delete every .pt under root except best.pt."""
    removed, freed = 0, 0

    for f in glob.glob(f"{root}/**/*.pt", recursive=True):
        if os.path.basename(f) == "best.pt":
            continue
        freed += os.path.getsize(f)
        os.remove(f)
        removed += 1
        print("removed:", f)

    print(f"\n{removed} files removed, {freed/1e6:.1f} MB freed")

    kept = glob.glob(f"{root}/**/best.pt", recursive=True)
    print(f"\nkept {len(kept)}:")
    for k in kept:
        print(" ", k, f"({os.path.getsize(k)/1e6:.1f} MB)")

    tot = sum(os.path.getsize(f) for f in glob.glob(f"{root}/**", recursive=True) if os.path.isfile(f))
    print(f"\n{tot/1e6:.1f} MB remaining in {root}")

keep_only_best()